In [ ]:
# #生成数据
# import numpy as np
# import networkx as nx

# n = 400
# p = 0.5

# # 生成随机图
# G = nx.gnp_random_graph(n, p)

# # 计算每个节点的邻居数量
# num_nb = np.array([len(list(G.neighbors(node))) for node in G.nodes()])

# # 将邻接矩阵进行归一化
# adj_matrix = nx.to_numpy_array(G)
# G = adj_matrix / np.sum(adj_matrix, axis=1, keepdims=True)

# print('G:', G)
# print('num_nb:', num_nb)

import numpy as np
import networkx as nx

#np.set_printoptions(threshold = np.inf)
#np.set_printoptions(suppress = True)

n = 600
degree_max = 5

# 生成度序列
degree_sequence = [np.random.randint(1, degree_max) for _ in range(n)]
# degree_sequence = np.ceil(degree_sequence)
# degree_sequence.astype(int)
while sum(degree_sequence) % 2 != 0:  # 确保度序列的总和是偶数
    degree_sequence = [np.random.randint(1, degree_max) for _ in range(n)]
#     degree_sequence = np.ceil(degree_sequence)
#     degree_sequence.astype(int)

G = nx.configuration_model(degree_sequence)

# 将多重边合并为单一边
G = nx.Graph(G)

# 生成邻接矩阵并计算邻居数量
adj_matrix = nx.to_numpy_array(G)
E = adj_matrix
num_nb = np.sum(E, axis=1)

# 计算G
G = E / np.sum(E, axis=1, keepdims=True)


print('G:', G)
print('E:', E)
print('num_nb:', num_nb)



In [ ]:
import numpy as np

# 对长度为n的正态分布随机数向量进行标准化
X = np.random.normal(size=n)
X = (X - np.mean(X)) / np.std(X, ddof=1)

# 生成长度为n的正态分布随机误差项epsilon
epsilon = np.random.normal(size=n)

# define the simulated outcomes and the exposure mapping function:
def get_Y(Z):
    return np.linalg.solve(-0.8*G+ np.diag([1]*n), -np.ones(n) + G.dot(Z) + Z + X + epsilon)

#这里可以换用多种仿真策略

def get_T(Z):
    return (E.dot(Z) > np.floor(num_nb/2)).astype(int)
#注意符号



In [ ]:
from scipy.stats import binom
import numpy as np

r1 = 0.5
# 计算 pscore1 和 pscore0
pscore1 = 1-binom.cdf(np.floor(num_nb/2), num_nb, r1)
pscore0 = 1 - pscore1

# print('binom.cdf(np.floor(num_nb/2), num_nb, r1):', binom.cdf(np.floor(1.5), 3, 0.5))

print('pscore1:', pscore1)
print('pscore0:', pscore0)

# 构建 A 矩阵
A = np.dot(E, E) > 0

# 计算 A_p 矩阵
temp = np.linalg.eig(A)
vectors = temp[1]
values = temp[0]
positive_values = values * (values > 0)
A_p = np.dot(np.dot(vectors, np.diag(positive_values)), np.linalg.inv(vectors))


In [ ]:
import numpy as np

def get_X(X, Z, G):
    return np.column_stack((Z, G.dot(Z), X, G.dot(X)))

def get_orth_coef(X, G, num_rep=10000):
    mom_mat = np.zeros((num_rep, 5))
    for i in range(num_rep):
        Z = np.random.binomial(1, r1, n)
        X_aug = get_X(X, Z, G)
        T_vec = get_T(Z)
        w = T_vec/pscore1 - (1 - T_vec)/pscore0
        mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

    mom_mean = np.mean(mom_mat, axis=0)
    return mom_mean[1:]/mom_mean[0]

orth_coef = get_orth_coef(X, G)



## add the intercept term!
def get_X_intercept(X, Z, G):
    return np.column_stack(( np.ones(len(X)), Z, G.dot(Z), X, G.dot(X)))


def get_orth_coef_intercept(X, G, num_rep=10000):
    mom_mat = np.zeros((num_rep, 6))
    for i in range(num_rep):
        Z = np.random.binomial(1, r1, n)
        X_aug = get_X_intercept(X, Z, G)
        T_vec = get_T(Z)
        w = T_vec/pscore1 - (1 - T_vec)/pscore0
        mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w), np.mean(X_aug[:, 4]*w)]

    mom_mean = np.mean(mom_mat, axis=0)
    return mom_mean[1:]/mom_mean[0]

orth_coef_intercept = get_orth_coef_intercept(X, G)










print('orth_coef:', orth_coef)
print('orth_coef_intercept:', orth_coef_intercept)

In [ ]:
# np.ceil([0.1,0.2])

# # np.random.randint(0, 10) for _ in range(n)
# degree_sequence = [np.random.uniform(0, 10) for _ in range(n)]
# # degree_sequence = np.ceil(degree_sequence)
# degree_sequence.astype(int)
# print(degree_sequence)

# 初始化一个空列表来存储计算结果
tau = []

# 进行10000次循环
for _ in range(10000):
    Z = np.random.binomial(1, r1, n)  # 生成服从二项分布的随机数Z
    Y = get_Y(Z)  # 根据Z计算Y
    T_vec = get_T(Z)  # 根据Z计算T_vec,分为exposure mapping中treated negighborhood是否过半的01两类
    
    estimate = Y*T_vec/pscore1 - Y*(1 - T_vec)/pscore0  # 根据Y、T_vec和pscore计算D
    tau_unadj_ht = np.mean(estimate)  # 计算未调整的tau值
    tau.append(tau_unadj_ht)  # 将tau_unadj_ht添加到列表中

# 计算所有tau_unadj_ht的均值
tau = np.mean(tau)


#这是NAIVE的估计：unadjusted HT estiamtor
# 打印均值
print(tau)


**CASE I (Baseline:) Unadjusted HT/Haj estimator**

In [ ]:
import scipy
from scipy.stats import binom, norm
# sim_res_1<- map_dfr(1:1000, ~{
#   Z <- rbinom(n, size = 1, prob = r1); Y <- get_Y(Z); X_aug <- 
#     (X,Z,G)
#   T_vec <- get_T(Z); D <- Y*T_vec/pscore1-Y*(1-T_vec)/pscore0; 
#   w <- T_vec/pscore1-(1-T_vec)/pscore0
    
#   tau_unadj_ht <- mean(D); 
  
#   var_est_unadj_1 <- t(D-tau_unadj_ht)%*%A%*%(D-tau_unadj_ht)/n^2 %>% as.vector();
#   var_est_unadj_2 <- t(D-tau_unadj_ht)%*%A_p%*%(D-tau_unadj_ht)/n^2 %>% as.vector()
    
#   is_cover_unadj_1 <- abs(tau_unadj_ht-tau)<qnorm(0.975)*sqrt(var_est_unadj_1)
#   is_cover_unadj_2 <- abs(tau_unadj_ht-tau)<qnorm(0.975)*sqrt(var_est_unadj_2)
  
#   return(tibble(tau_unadj_ht, var_est_unadj_1, var_est_unadj_2, is_cover_unadj_1, is_cover_unadj_2 ))
# })

import numpy as np
import pandas as pd

###Here is the baseline.

sim_res_1 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # 生成服从二项分布的随机数Z
#     print('Z', Z)
    Y = get_Y(Z)  # 根据Z计算Y
#     print('X:',X)
#     print('Z:', Z)
#     print('G:', G)

#     X_aug = get_X(X, Z, G)  # 生成X_aug（应该是X, Z, G的组合）
    T_vec = get_T(Z)  # 根据Z计算T_vec
    D = Y*T_vec/pscore1 - Y*(1 - T_vec)/pscore0  # 根据Y、T_vec和pscore计算D
#     print('D:', D)

    w = T_vec/pscore1 - (1 - T_vec)/pscore0
    tau_unadj_ht = np.mean(D)  # 计算未调整的tau值
#     print('tau_unadj_ht:', tau_unadj_ht)
    
    var_est_unadj_1 = np.dot(np.dot((D-tau_unadj_ht).T, A), (D-tau_unadj_ht)) / n**2
#     print('var_est_unadj_1:',  var_est_unadj_1)
    var_est_unadj_2 = np.dot(np.dot((D-tau_unadj_ht).T, A_p), (D-tau_unadj_ht)) / n**2
# condition = np.abs(tau_unadj_ht - tau) < scipy.stats.norm.ppf(0.975) * np.sqrt(var_est_unadj_1)
# result = np.where(condition, 1, other_value)
    is_cover_unadj_1 = np.abs(tau_unadj_ht - tau) < norm.ppf(0.975) * np.sqrt(var_est_unadj_1)
#     print('is_cover_unadj_1:', is_cover_unadj_1.shape)
#     print('norm.ppf(0.975) :', norm.ppf(0.975) )
    is_cover_unadj_2 = np.abs(tau_unadj_ht - tau) < norm.ppf(0.975) * np.sqrt(var_est_unadj_2)
    
    sim_res_1.append([tau_unadj_ht, var_est_unadj_1, var_est_unadj_2, is_cover_unadj_1, is_cover_unadj_2])

sim_res_1_df_ht = pd.DataFrame(sim_res_1, columns=['tau_unadj_ht', 'var_est_unadj_1', 'var_est_unadj_2', 'is_cover_unadj_1', 'is_cover_unadj_2'])

print('sim_res_1_df_ht:', sim_res_1_df_ht)

print('estimate_ht:',    sim_res_1_df_ht['tau_unadj_ht'].mean())
print('variance_A_ht:',  sim_res_1_df_ht['var_est_unadj_1'].mean())
print('variance_Ap_ht:', sim_res_1_df_ht['var_est_unadj_2'].mean())
print('coverage_A_ht:',  np.array(sim_res_1_df_ht['is_cover_unadj_1']).mean())
print('coverage_Ap_ht:', np.array(sim_res_1_df_ht['is_cover_unadj_2']).mean())


##The above is the HT-estimator, the below is the Haj-estimator:

sim_res_1 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # 生成服从二项分布的随机数Z
#     print('Z', Z)
    Y = get_Y(Z)  # 根据Z计算Y
#     print('X:',X)
#     print('Z:', Z)
#     print('G:', G)

#     X_aug = get_X(X, Z, G)  # 生成X_aug（应该是X, Z, G的组合）
    T_vec = get_T(Z)  # 根据Z计算T_vec
#     D = Y*T_vec/pscore1 - Y*(1 - T_vec)/pscore0  # 根据Y、T_vec和pscore计算D
#     print('D:', D)
    D = (Y * T_vec/pscore1).sum()/(T_vec/pscore1).sum() - (Y * (1-T_vec)/pscore0).sum()/ ((1-T_vec)/pscore0).sum()
    
    
    w = T_vec/pscore1 - (1 - T_vec)/pscore0
    tau_unadj_haj = D  # 计算未调整的tau值, 这是haj估计器
#     print('tau_unadj_ht:', tau_unadj_ht)
    
    tempt = (Y * T_vec - (Y * T_vec).mean())/pscore1 - (Y * (1-T_vec) - (Y * (1-T_vec)).mean())/pscore0
    var_est_unadj_1 = np.dot(np.dot(tempt, A), tempt) / n**2
    var_est_unadj_2 = np.dot(np.dot(tempt, A_p), tempt)/ n**2
# condition = np.abs(tau_unadj_ht - tau) < scipy.stats.norm.ppf(0.975) * np.sqrt(var_est_unadj_1)
# result = np.where(condition, 1, other_value)
    is_cover_unadj_1 = np.abs(tau_unadj_haj - tau) < norm.ppf(0.975) * np.sqrt(var_est_unadj_1)
#     print('is_cover_unadj_1:', is_cover_unadj_1.shape)
#     print('norm.ppf(0.975) :', norm.ppf(0.975) )
    is_cover_unadj_2 = np.abs(tau_unadj_haj - tau) < norm.ppf(0.975) * np.sqrt(var_est_unadj_2)
    
    sim_res_1.append([tau_unadj_haj, var_est_unadj_1, var_est_unadj_2, is_cover_unadj_1, is_cover_unadj_2])

sim_res_1_df_haj = pd.DataFrame(sim_res_1, columns=['tau_unadj_haj', 'var_est_unadj_1', 'var_est_unadj_2', 'is_cover_unadj_1', 'is_cover_unadj_2'])







print('sim_res_1_df_haj:', sim_res_1_df_haj)

print('estimate_haj:',    sim_res_1_df_haj['tau_unadj_haj'].mean())
print('variance_A_haj:',  sim_res_1_df_haj['var_est_unadj_1'].mean())
print('variance_Ap_haj:', sim_res_1_df_haj['var_est_unadj_2'].mean())
print('coverage_A_haj:',  np.array(sim_res_1_df_haj['is_cover_unadj_1']).mean())
print('coverage_Ap_haj:', np.array(sim_res_1_df_haj['is_cover_unadj_2']).mean())




















**Our regression (6 cases: HT/Haj + pointwise/jointly + interaction/or not without considering interaction+pointesie decorrelation)**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import binom, norm
from sklearn.preprocessing import scale


#PNI: point + not iter: G_dc^1,
#point + iter: none
#JNI: jointly + not iter: G_dc^2
#JI: jointly + iter: G_dc^2(t,t') 


#CASE I: HT estimator: 1 point + not iter


sim_res_2 = []



def our_estimator(X_db, X_aug, w, A, A_p):
   
    
    D_2 = scale(X_db * np.array(w).reshape(-1,1), axis=0, with_std=False)
#     print('D_2:', D_2)
    hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
    hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

    tau_adj_aug1 = np.mean((Y - np.dot(X_db, hbeta_1)) * w)
    tau_adj_aug2 = np.mean((Y - np.dot(X_db, hbeta_2)) * w)
    
#     bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau_adj_aug1
#     bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau_adj_aug2
#     print('bias1:', bias1)
#     print('bias2:', bias2)

    var_est_adj_aug1 = np.dot(np.dot((D - np.dot(D_2, hbeta_1) - tau_adj_aug1).T, A), (D - np.dot(D_2, hbeta_1) - tau_adj_aug1)) / n**2
    var_est_adj_aug2 = np.dot(np.dot((D - np.dot(D_2, hbeta_2) - tau_adj_aug2).T, A_p), (D - np.dot(D_2, hbeta_2) - tau_adj_aug2)) / n**2

    is_cover_adj_aug_1 = abs(tau_adj_aug1 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug1)
    is_cover_adj_aug_2 = abs(tau_adj_aug2 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug2)
#     print('is_cover_adj_aug_1:', is_cover_adj_aug_1.shape)
    
    sim_res_2.append([tau_adj_aug1, tau_adj_aug2, var_est_adj_aug1, var_est_adj_aug2, is_cover_adj_aug_1, is_cover_adj_aug_2])

##here is the HT-estimator:
bias1_set = []
bias2_set = []

for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X_intercept(X, Z, G)  # HT的第一种表征,带有截距

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    tau_unadj_ht = np.mean(D)
    
    X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), orth_coef_intercept.reshape(1, -1))
    our_estimator(X_db_PNI, X_aug, w, A, A_p)
    #额外打印bias,不做decorrelated 的情形：
    D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
#     print('D_2:', D_2)
    hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
    hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

   
    
    bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau
    bias1_set.append(bias1)
    bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau
    bias2_set.append(bias2)
    print('bias1:', np.array(bias1_set).mean())
    print('bias2:', np.array(bias2_set).mean())
    #额外打印bias,不做decorrelated 的情形
    
    
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])

print('sim_res_2_df:', sim_res_2_df)
print('estimate1_ht_PNI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_ht_PNI:', sim_res_2_df['tau_adj_aug2'].mean())


print('variance_1_ht_PNI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_ht_PNI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_ht_PNI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_ht_PNI:', proportion_true)





##运行完成后清空，开始JNI; 注意这里出现了一个问题：如果多使用了截距项的话，会导致\beta求解的时候singular matrix！
sim_res_2 = []



mu_1_list = []
mu_0_list = []
for _ in range(1000):
    Z = np.random.binomial(1, r1,n) 
    T_vec = get_T(Z)
    X_aug = get_X(X, Z, G)  # HT的第一种表征,带有截距
    T_vec = np.array(T_vec).reshape(-1,1) 
    p1 = np.array(pscore1).reshape(-1,1)
    column_sums_1 = np.sum( X_aug * T_vec/p1 , axis=0)
    mu_1 = column_sums_1.reshape(-1, 1)/n
    mu_1_list.append(mu_1)
        
    p0 = np.array(pscore0).reshape(-1,1)
    column_sums_0 = np.sum(X_aug * (1-T_vec)/p0 , axis=0)
    mu_0 = column_sums_0.reshape(-1, 1)/n
    mu_0_list.append(mu_0)
    
column_sums_1 = np.sum( mu_1_list , axis=0)
mu_1 = column_sums_1.reshape(-1, 1)/1000
column_sums_0 = np.sum( mu_0_list , axis=0)
mu_0 = column_sums_0.reshape(-1, 1)/1000


    
    
    


for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    tau_unadj_ht = np.mean(D)
  


    
    T_vec = np.array(T_vec).reshape(-1,1) 
  
    #column_sums_1 = np.sum( X_aug * T_vec , axis=0)
#     mu_1 = column_sums_1.reshape(-1, 1)/T_vec.sum()

#     column_sums_0 = np.sum(X_aug * (1-T_vec) , axis=0)
#     mu_0 = column_sums_0.reshape(-1, 1)/(1-T_vec).sum()


    X_db_JNI = X_aug - np.dot(T_vec, mu_1.T) - np.dot((1-T_vec), mu_0.T)
#     print('X_db_JNI :', X_db_JNI )
#     print('X_db_JNI.shape:', X_db_JNI.shape)
    
    
    our_estimator(X_db_JNI, X_aug, w, A, A_p)
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])
print('sim_res_2_df:', sim_res_2_df)


print('estimate1_ht_JNI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_ht_JNI:', sim_res_2_df['tau_adj_aug2'].mean())

print('variance_1_ht_JNI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_ht_JNI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_ht_JNI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_ht_JNI:', proportion_true)


#finally,我们进行JI的测试：

sim_res_2 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  # HT的第一种表征,带有截距

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    tau_unadj_ht = np.mean(D)
  


    
    T_vec = np.array(T_vec).reshape(-1,1) 
    

#     column_sums_1 = np.sum( X_aug * T_vec , axis=0)
#     mu_1 = column_sums_1.reshape(-1, 1)/T_vec.sum()
#     column_sums_0 = np.sum(X_aug * (1-T_vec) , axis=0)
#     mu_0 = column_sums_0.reshape(-1, 1)/(1-T_vec).sum()


    X_db_JNI = X_aug - np.dot(T_vec, mu_1.T) - np.dot((1-T_vec), mu_0.T)
    print('X_db_JNI :', X_db_JNI )
    print('X_db_JNI.mean :', X_db_JNI.mean() )
    print('X_db_JNI.shape:', X_db_JNI.shape)
    
    
    
    
    
    
    our_estimator( np.column_stack((X_db_JNI * T_vec, X_db_JNI * (1-T_vec))), X_aug, w, A, A_p)
    
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])
print('sim_res_2_df:', sim_res_2_df)
print('estimate1_ht_JI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_ht_JI:', sim_res_2_df['tau_adj_aug2'].mean())

print('variance_1_ht_JI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_ht_JI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_ht_JI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_ht_JI:', proportion_true)













































In [ ]:
print('X_db_JNI * w:', (X_db_JNI * np.array(w).reshape(-1,1)).sum())
print(hbeta_1)
print(hbeta_2)
print(mu_1)
print(mu_0)

**Ding's regression (Lin's regression & Fisher's regression)**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import binom, norm
from sklearn.preprocessing import scale


#PNI: point + not iter: G_dc^1,
#point + iter: none
#JNI: jointly + not iter: G_dc^2
#JI: jointly + iter: G_dc^2(t,t') 


#CASE I: Haj estimator: 1 point + not iter


sim_res_2 = []



def our_estimator(X_db, X_aug, w, A, A_p):
   
    
    D_2 = scale(X_db * np.array(w).reshape(-1,1), axis=0, with_std=False)
#     print('D_2:', D_2)
    hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
    hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

    tau_adj_aug1 = np.mean((Y - np.dot(X_db, hbeta_1)) * w)
    tau_adj_aug2 = np.mean((Y - np.dot(X_db, hbeta_2)) * w)
    
#     bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau_adj_aug1
#     bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau_adj_aug2
#     print('bias1:', bias1)
#     print('bias2:', bias2)

    var_est_adj_aug1 = np.dot(np.dot((D - np.dot(D_2, hbeta_1) - tau_adj_aug1).T, A), (D - np.dot(D_2, hbeta_1) - tau_adj_aug1)) / n**2
    var_est_adj_aug2 = np.dot(np.dot((D - np.dot(D_2, hbeta_2) - tau_adj_aug2).T, A_p), (D - np.dot(D_2, hbeta_2) - tau_adj_aug2)) / n**2

    is_cover_adj_aug_1 = abs(tau_adj_aug1 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug1)
    is_cover_adj_aug_2 = abs(tau_adj_aug2 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug2)
#     print('is_cover_adj_aug_1:', is_cover_adj_aug_1.shape)
    
    sim_res_2.append([tau_adj_aug1, tau_adj_aug2, var_est_adj_aug1, var_est_adj_aug2, is_cover_adj_aug_1, is_cover_adj_aug_2])

##here is the Haj-estimator:
bias1_set = []
bias2_set = []

for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X_intercept(X, Z, G)  # HT的第一种表征,带有截距

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    
#     w = T_vec / pscore1 - (1 - T_vec) / pscore0
    
    w = ( T_vec / pscore1 )/ (1/n*np.sum(T_vec / pscore1)) -  ( (1-T_vec) / pscore0 )/ (1/n*np.sum((1-T_vec) / pscore0))
    
    tau_unadj_ht = np.mean(D)
    
    X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), orth_coef_intercept.reshape(1, -1))
    our_estimator(X_db_PNI, X_aug, w, A, A_p)
    #额外打印bias,不做decorrelated 的情形：
    D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
#     print('D_2:', D_2)
    hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
    hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

   
    
    bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau
    bias1_set.append(bias1)
    bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau
    bias2_set.append(bias2)
    print('bias1:', np.array(bias1_set).mean())
    print('bias2:', np.array(bias2_set).mean())
    #额外打印bias,不做decorrelated 的情形
    
    
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])

print('sim_res_2_df:', sim_res_2_df)
print('estimate1_haj_PNI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_haj_PNI:', sim_res_2_df['tau_adj_aug2'].mean())


print('variance_1_haj_PNI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_haj_PNI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_haj_PNI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_haj_PNI:', proportion_true)





##运行完成后清空，开始JNI; 注意这里出现了一个问题：如果多使用了截距项的话，会导致\beta求解的时候singular matrix！
sim_res_2 = []



mu_1_list = []
mu_0_list = []
for _ in range(1000):
    Z = np.random.binomial(1, r1,n) 
    T_vec = get_T(Z)
    X_aug = get_X(X, Z, G)  # HT的第一种表征,带有截距
    T_vec = np.array(T_vec).reshape(-1,1) 
    p1 = np.array(pscore1).reshape(-1,1)
    column_sums_1 = np.sum( X_aug * T_vec/p1 , axis=0)
    mu_1 = column_sums_1.reshape(-1, 1)/n
    mu_1_list.append(mu_1)
        
    p0 = np.array(pscore0).reshape(-1,1)
    column_sums_0 = np.sum(X_aug * (1-T_vec)/p0 , axis=0)
    mu_0 = column_sums_0.reshape(-1, 1)/n
    mu_0_list.append(mu_0)
    
column_sums_1 = np.sum( mu_1_list , axis=0)
mu_1 = column_sums_1.reshape(-1, 1)/1000
column_sums_0 = np.sum( mu_0_list , axis=0)
mu_0 = column_sums_0.reshape(-1, 1)/1000


    
    
    


for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
#     w = T_vec / pscore1 - (1 - T_vec) / pscore0
    w = ( T_vec / pscore1 )/ (1/n*np.sum(T_vec / pscore1)) -  ( (1-T_vec) / pscore0 )/ (1/n*np.sum((1-T_vec) / pscore0))
    tau_unadj_ht = np.mean(D)
  


    
    T_vec = np.array(T_vec).reshape(-1,1) 
  
    #column_sums_1 = np.sum( X_aug * T_vec , axis=0)
#     mu_1 = column_sums_1.reshape(-1, 1)/T_vec.sum()

#     column_sums_0 = np.sum(X_aug * (1-T_vec) , axis=0)
#     mu_0 = column_sums_0.reshape(-1, 1)/(1-T_vec).sum()


    X_db_JNI = X_aug - np.dot(T_vec, mu_1.T) - np.dot((1-T_vec), mu_0.T)
#     print('X_db_JNI :', X_db_JNI )
#     print('X_db_JNI.shape:', X_db_JNI.shape)
    
    
    our_estimator(X_db_JNI, X_aug, w, A, A_p)
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])
print('sim_res_2_df:', sim_res_2_df)


print('estimate1_haj_JNI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_haj_JNI:', sim_res_2_df['tau_adj_aug2'].mean())

print('variance_1_haj_JNI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_haj_JNI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_haj_JNI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_haj_JNI:', proportion_true)


#finally,我们进行JI的测试：

sim_res_2 = []

for _ in range(1000):
    Z = np.random.binomial(1, r1,n)  # rbinom 函数的Python等价代码
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
    X_aug = get_X(X, Z, G)  # HT的第一种表征,带有截距

    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    
#     print('T_vec:', T_vec)
#     print('T_vec.shape:', (np.array(T_vec)).reshape(-1,1))
#     print('X_aug:', X_aug)
#     print('X_aug.shape:', X_aug.shape)
    
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
#     w = T_vec / pscore1 - (1 - T_vec) / pscore0
    w = ( T_vec / pscore1 )/ (1/n*np.sum(T_vec / pscore1)) -  ( (1-T_vec) / pscore0 )/ (1/n*np.sum((1-T_vec) / pscore0))
    tau_unadj_ht = np.mean(D)
  


    
    T_vec = np.array(T_vec).reshape(-1,1) 
    

#     column_sums_1 = np.sum( X_aug * T_vec , axis=0)
#     mu_1 = column_sums_1.reshape(-1, 1)/T_vec.sum()
#     column_sums_0 = np.sum(X_aug * (1-T_vec) , axis=0)
#     mu_0 = column_sums_0.reshape(-1, 1)/(1-T_vec).sum()


    X_db_JNI = X_aug - np.dot(T_vec, mu_1.T) - np.dot((1-T_vec), mu_0.T)
    print('X_db_JNI :', X_db_JNI )
    print('X_db_JNI.mean :', X_db_JNI.mean() )
    print('X_db_JNI.shape:', X_db_JNI.shape)
    
    
    
    
    
    
    our_estimator( np.column_stack((X_db_JNI * T_vec, X_db_JNI * (1-T_vec))), X_aug, w, A, A_p)
    
    
sim_res_2_df = pd.DataFrame(sim_res_2, columns=['tau_adj_aug1', 'tau_adj_aug2', 'var_est_adj_aug1', 'var_est_adj_aug2', 'is_cover_adj_aug_1', 'is_cover_adj_aug_2'])
print('sim_res_2_df:', sim_res_2_df)
print('estimate1_haj_JI:', sim_res_2_df['tau_adj_aug1'].mean())
print('estimate2_haj_JI:', sim_res_2_df['tau_adj_aug2'].mean())

print('variance_1_haj_JI:', sim_res_2_df['var_est_adj_aug1'].mean())
print('variance_2_haj_JI:', sim_res_2_df['var_est_adj_aug2'].mean())

proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_1']).mean()
print('coverage1_haj_JI:', proportion_true)
proportion_true =  np.array(sim_res_2_df['is_cover_adj_aug_2']).mean()
print('coverage2_haj_JI:', proportion_true)













































In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import binom, norm
import statsmodels.api as sm



# sim_res_3<- map_dfr(1:1000, ~{
#   Z <- rbinom(n, size = 1, prob = r1); Y <- get_Y(Z); X_aug <- 
#     (X,Z,G)
#   T_vec <- get_T(Z); D <- Y*T_vec/pscore1-Y*(1-T_vec)/pscore0; 
#   w <- T_vec/pscore1-(1-T_vec)/pscore0
  
#   lm_haj <- lm(Y~1+T_vec+X+T_vec:X, w = T_vec/pscore1+(1-T_vec)/pscore0)
#   e_haj <- lm_haj %>% resid(); tau_adj_haj <- lm_haj %>% coef() %>% .[2]
#   C_haj <- cbind(1,T_vec,X,T_vec*X)
#   var_est_adj_haj <- solve(t(C_haj)%*%diag(w)%*%C_haj)%*%(t(C_haj)%*%diag(w)%*%diag(e_haj)%*%A_p%*%diag(e_haj)%*%diag(w)%*%(C_haj))%*%solve(t(C_haj)%*%diag(w)%*%C_haj) %>% .[2,2]
#   is_cover_adj_haj <- abs(tau_adj_haj-tau)<qnorm(0.975)*sqrt(var_est_adj_haj)
  
#   return(tibble(tau_adj_haj, var_est_adj_haj , is_cover_adj_haj))
# })


sim_res_3 = []


for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # rbinom 函数的Python等价代码
    print('Z:',Z)
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
#     X_aug = get_X(X, Z, G)  # 合并X, Z, G的数据，得到X_aug
    print('Y:',Y)
    T_vec = get_T(Z)  # get_T 函数的Python等价代码
    print('T_vec1:',T_vec)
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 - (1 - T_vec) / pscore0
    print('w.shape:', w.shape)
    #w = [1]* n
    w = np.array(w)

    print('T_vec2:',T_vec)
    # 将数据合并到一个DataFrame中
    data = pd.DataFrame({'cons': np.ones(len(X)), 'Y': Y, 'T_vec': T_vec, 'X': X, 'Interaction': T_vec*X, 'w': T_vec/pscore1+(1-T_vec)/pscore0})

    # 处理NAN值，例如使用均值填充
    data = data.dropna()

    # 拟合线性回归模型，w作为权重传递给OLS回归模型
    lm_haj = sm.WLS(data['Y'], sm.add_constant(data[['cons', 'T_vec', 'X', 'Interaction']]), weights=data['w']).fit()
    print('T_vec3:',T_vec)

    
    e_haj = lm_haj.resid
    tau_adj_haj = lm_haj.params[1]  # 获取模型参数
    print('lm_haj:', lm_haj)
    print('lm_haj_param:', lm_haj.params)
    print('lm_haj_param[0]:', lm_haj.params[0])
    print('lm_haj_param[1]:', lm_haj.params[1])
    print('lm_haj_param[2]:', lm_haj.params[2])
    print('lm_haj_param[3]:', lm_haj.params[3])
    print('T_vec4:',T_vec.shape)
    C_haj = np.column_stack( ( (np.ones(len(X))), T_vec, X, T_vec * X) )
#     var_est_adj_haj = np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.linalg.inv(np.dot(np.dot(C_haj.T, np.diag(w)), C_haj)),
#                                                       C_haj.T), np.diag(w)), np.diag(e_haj)), A_p), np.diag(e_haj)), np.diag(w)), C_haj))
    print('np.ones(len(X)).T:', np.ones(len(X)).T)
    print('T_vec:', T_vec)
    print('X:', X)
    print('np.array(T_vec).reshape(-1,1)*X):', np.array(T_vec).reshape(-1,1)*X)
    print('C_haj:', C_haj.shape)
    temp_term = np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj) @ (C_haj.T @ np.diag(w) @ np.diag(e_haj) @ A_p @ np.diag(e_haj) @ np.diag(w) @ C_haj) @ np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj)
    
    print('temp_term:', temp_term)
    print('temp_term[1,1]:', temp_term[1,1])
    print('temp_term_shape:', temp_term.shape)
    var_est_adj_haj = temp_term[1, 1]

    is_cover_adj_haj = abs(tau_adj_haj - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_haj)
    sim_res_3.append([tau_adj_haj, var_est_adj_haj, is_cover_adj_haj])

sim_res_3_df = pd.DataFrame(sim_res_3, columns=['tau_adj_haj', 'var_est_adj_haj', 'is_cover_adj_haj'])

print('sim_res_3_df_Lin:', sim_res_3_df)
print('estimate_F_Lin:', sim_res_3_df['tau_adj_haj'].mean())
print('variance_F_Lin:', sim_res_3_df['var_est_adj_haj'].mean())
print('coverage_F_Lin:', np.array(sim_res_3_df['is_cover_adj_haj']).mean())


###the above is the lin's regression; the below is the fisher's regression:

sim_res_3 = []

temp_term = []
e_haj = []
C_haj = []
w = []
data = []

for _ in range(1000):
    Z = np.random.binomial(1, r1, n)  # rbinom 函数的Python等价代码
#     print('Z:',Z)
    Y = get_Y(Z)  # get_Y 函数的Python等价代码
#     X_aug = get_X(X, Z, G)  # 合并X, Z, G的数据，得到X_aug
#     print('Y:',Y)
    T_vec = get_T(Z)  # get_T 函数的Python等价代码
#     print('T_vec1:',T_vec)
    D = Y * T_vec / pscore1 - Y * (1 - T_vec) / pscore0
    w = T_vec / pscore1 + (1 - T_vec) / pscore0    #这是回归的weight
#     print('w.shape:', w.shape)
    #w = [1]* n
    w = np.array(w)

#     print('T_vec2:',T_vec)
    # 将数据合并到一个DataFrame中
    data = pd.DataFrame({'cons': np.ones(len(X)), 'Y': Y, 'T_vec': T_vec, 'X': X, 'w': T_vec/pscore1+(1-T_vec)/pscore0})

    # 处理NAN值，例如使用均值填充
    data = data.dropna()

    # 拟合线性回归模型，w作为权重传递给OLS回归模型
    lm_haj = sm.WLS(data['Y'], sm.add_constant(data[['cons', 'T_vec', 'X']]), weights=data['w']).fit()
#     print('T_vec3:',T_vec)

    
    e_haj = lm_haj.resid
    tau_adj_haj = lm_haj.params[1]  # 获取模型参数
#     print('lm_haj:', lm_haj)
#     print('lm_haj_param:', lm_haj.params)
#     print('lm_haj_param[0]:', lm_haj.params[0])
#     print('lm_haj_param[1]:', lm_haj.params[1])
#     print('lm_haj_param[2]:', lm_haj.params[2])
#     print('T_vec4:',T_vec.shape)
    C_haj = np.column_stack( ( np.ones(len(X)), T_vec, X) )  #here there are only three terms
    
#     var_est_adj_haj = np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.dot(np.linalg.inv(np.dot(np.dot(C_haj.T, np.diag(w)), C_haj)),
#                                                       C_haj.T), np.diag(w)), np.diag(e_haj)), A_p), np.diag(e_haj)), np.diag(w)), C_haj))
#     print('np.ones(len(X)).T:', np.ones(len(X)).T)
#     print('T_vec:', T_vec)
#     print('X:', X)
#     print('np.array(T_vec).reshape(-1,1)*X):', np.array(T_vec).reshape(-1,1)*X)
#     print('C_haj:', C_haj.shape)
    temp_term = np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj) @ (C_haj.T @ np.diag(w) @ np.diag(e_haj) @ A_p @ np.diag(e_haj) @ np.diag(w) @ C_haj) @ np.linalg.inv(C_haj.T @ np.diag(w) @ C_haj)
    
#     print('temp_term:', temp_term)
#     print('temp_term[1,1]:', temp_term[1,1])
#     print('temp_term_shape:', temp_term.shape)
    var_est_adj_haj = temp_term[1, 1]

    is_cover_adj_haj = abs(tau_adj_haj - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_haj)
    sim_res_3.append([tau_adj_haj, var_est_adj_haj, is_cover_adj_haj])

sim_res_3_df = pd.DataFrame(sim_res_3, columns=['tau_adj_haj', 'var_est_adj_haj', 'is_cover_adj_haj'])

print('sim_res_3_df_fisher:', sim_res_3_df)
print('estimate_F_fisher:', sim_res_3_df['tau_adj_haj'].mean())
print('variance_F_fisher:', sim_res_3_df['var_est_adj_haj'].mean())
print('coverage_F_fisher:', np.array(sim_res_3_df['is_cover_adj_haj']).mean())



In [ ]:
print('mu_1:', mu_1)
print('mu_0:', mu_0)